In [2]:
zones = {
    "takeoff": ["W1", "W2", "W3", "W4"],
    "mid":     ["W5", "W6", "W7", "W8"],
    "landing": ["W9", "W10", "W11", "W12"]
}

wind_features = ["Speed", "Tangent", "Cross", "Turbulence"]

In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import math

def plot(X_seq, X_sim, j, name):
    """
    Plot actual vs simulated trajectory in 3D with projections.
    Args:
        X_seq (ndarray): ground truth state sequence (T, state_dim)
        X_sim (ndarray): simulated state sequence (T, state_dim)
        j (int): index of jump
        name (str): identifier for saving plots """

    save_dir = f"plots/{name}"
    os.makedirs(save_dir, exist_ok=True)

    
    actual = X_seq[:, :3]   #vzame X,Y,Z iz [x, y, z, vx, vy, vz]
    sim = X_sim[:, :3]          # same iz simulacije

    Xa, Ya, Za = actual[:,0], actual[:,1], actual[:,2]
    Xs, Ys, Zs = sim[:,0], sim[:,1], sim[:,2]

    napaka = np.mean(error_fun(actual, sim))
    length = (Xs[-1]**2 + Ys[-1]**2 + Zs[-1]**2)**(1/2)
    length_a = (Xa[-1]**2 + Ya[-1]**2 + Za[-1]**2)**(1/2)

    fig = plt.figure(figsize=(8, 5))
    ax = fig.add_subplot(111, projection='3d')

    ax.plot3D(Xa, Ya, Za, label="Actual", color="blue")
    ax.plot3D(Xs, Ys, Zs, label="Simulated", color="red", linestyle="--")


    # Ground projections
    ax.plot(Xa, 15, Za, color="blue", alpha=0.3, linestyle=':')
    ax.plot(Xs, 15, Zs, color="red", alpha=0.3, linestyle=':')

    ax.plot(0, Ya, Za, color="blue", alpha=0.3, linestyle=':')
    ax.plot(0, Ys, Zs, color="red", alpha=0.3, linestyle=':')


    ax.set_title(f"Actual vs Simulated Trajectory\n Error: {napaka:.3f}, Simulated length: {length:.1f},  Actual length: {length_a:.1f}")
    ax.set_xlabel("X [m]")
    ax.set_ylabel("Y [m]")
    ax.set_zlabel("Z [m]")
    ax.legend()
    ax.set_box_aspect([1,1,1])
    ax.set_ylim(-15, 15)

    plt.legend()
    #plt.tight_layout()

    filename = f"2SSM flight_simulation{j + 1}.png"
    plt.savefig(os.path.join(save_dir, filename), dpi=300)
    plt.close(fig)
    #plt.show()


### old functions

In [15]:
import numpy as np
from scipy.interpolate import interp1d

def resample_curve(curve, n_points=200):
    """
    Resample a 3D curve to have exactly n_points, parameterized by arc length.

    Args:
        curve: (N, d) array, trajectory points
        n_points: number of resampled points

    Returns:
        (n_points, d) array, resampled curve
    """
    # compute arc length
    diffs = np.diff(curve, axis=0)
    seg_lengths = np.linalg.norm(diffs, axis=1)
    arc = np.concatenate([[0], np.cumsum(seg_lengths)])

    # normalize arc length to [0, 1]
    arc_norm = arc / arc[-1]

    # create interpolators
    resampled = []
    t_new = np.linspace(0, 1, n_points)
    for d in range(curve.shape[1]):
        f = interp1d(arc_norm, curve[:, d], kind="linear")
        resampled.append(f(t_new))
    return np.stack(resampled, axis=1)

In [16]:
def curve_error_with_overshoot(curve_a, curve_b, n_points=200):
    # resample both
    len_a = len(curve_a)
    len_b = len(curve_b)

    n_common = min(len_a, len_b)
    a_res = resample_curve(curve_a, n_common)
    b_res = resample_curve(curve_b, n_common)

    # pointwise error on common part
    errors = list(np.linalg.norm(a_res - b_res, axis=1))

    # overshoot penalty
    if len_a > len_b:
        last_b = curve_b[-1]
        extra = curve_a[len_b:]
        for p in extra:
            errors.append(np.linalg.norm(p - last_b))
    elif len_b > len_a:
        last_a = curve_a[-1]
        extra = curve_b[len_a:]
        for p in extra:
            errors.append(np.linalg.norm(p - last_a))

    return np.mean(errors), np.array(errors)


### new funtions

In [4]:
def interpolate_curve_by_x(curve):
    """
    Interpolates a 3D curve by x-coordinate and returns (x, y, z),
    keeping the original start and end points exactly.

    Args:
        curve : array-like of shape (N, 3)
            Each row is [x, y, z].

    Returns:
        result : ndarray of shape (M, 3)
            Interpolated coordinates at integer x plus original endpoints.
    """

    curve = np.asarray(curve, dtype=float)
    if curve.ndim != 2 or curve.shape[1] != 3:
        raise ValueError("sth")

    curve = curve[np.argsort(curve[:, 0])]

    x = curve[:, 0]
    y = curve[:, 1]
    z = curve[:, 2]    

    x_new = np.arange(np.ceil(x.min()) + 0, np.floor(x.max()) + 1)

    f_y = interp1d(x, y, kind="linear", bounds_error=False, fill_value="extrapolate")
    f_z = interp1d(x, z, kind="linear", bounds_error=False, fill_value="extrapolate")

    if len(x_new) > 0:
        y_new = f_y(x_new)
        z_new = f_z(x_new)
        result = np.vstack([
            curve[0],
            np.column_stack((x_new, y_new, z_new)),
            curve[-1]
        ])
    else:
        result = curve

    return result


In [5]:
def error_fun(curveA, curveB):
    curve1 = interpolate_curve_by_x(curveA)
    curve2 = interpolate_curve_by_x(curveB)

    if len(curve1) < len(curve2):
        longer = curve2
        shorter = curve1
    else:
        longer = curve1
        shorter = curve2

    longer1 = longer[:len(shorter)]
    longer2 = longer[len(shorter):]

    shorter2 = np.ones(len(longer2)) * shorter[-1]

    error = np.linalg.norm(longer1, shorter) + np.linalg.norm(shorter2, longer2)

    return error

    

In [6]:


def preprocess_flight_normalized(df, step=0.05):
    """
    Preprocess a normalized flight dataframe (already interpolated) to generate states (X), observations (Y), and controls (U).
    
    Args:
        df (pd.DataFrame): Normalized flight dataframe (fixed time step).
        step (float): Time step between rows (used for derivatives).
        
    Returns:
        states (np.ndarray): State matrix [time_steps, state_dim].
        observations (np.ndarray): Observation matrix [time_steps, obs_dim].
        controls (np.ndarray): Control matrix [time_steps, control_dim].
    """

    df.columns = df.columns.str.strip()   #izloči imena stolpcev
    df = df.iloc[1:]
    
    df = df.ffill().bfill()   #back fill za manjkajoče vrednosti
    
    x = df["X [m]"].to_numpy()   #save values
    y = df["Y [m]"].to_numpy()
    z = df["Z [m]"].to_numpy()
    
    dt = step
    #vx = np.gradient(x, dt)   #gradient za hitrost
    vx = df.get("Speed hor. [km/h]", pd.Series([0]*len(x))).to_numpy() * (100/36)
    vy = np.gradient(y, dt)
    #vz = np.gradient(z, dt)
    vz = df.get("Speed ver. [km/h]", pd.Series([0]*len(x))).to_numpy() * (100/36)

    speed = df.get("speed resulting [km/h]", pd.Series([0]*len(x))).to_numpy() * (100/36)

    opening = df.get("Opening Angle [°]", pd.Series([0]*len(x))).to_numpy()
    roll_L = df.get("Roll Angle Left [°]", pd.Series([0]*len(x))).to_numpy()
    roll_R = df.get("Roll Angle Right [°]", pd.Series([0]*len(x))).to_numpy()
    yaw_L = df.get("Yaw Angle Left [°]", pd.Series([0]*len(x))).to_numpy()
    yaw_R = df.get("Yaw Angle Right [°]", pd.Series([0]*len(x))).to_numpy()
    stall_L = df.get("Stalling Angle Left [°]", pd.Series([0]*len(x))).to_numpy()
    stall_R = df.get("Stalling Angle Right [°]", pd.Series([0]*len(x))).to_numpy()

    states = np.stack([x, y, z, vx, vy, vz, speed, opening, roll_L, roll_R, yaw_L, yaw_R, stall_L, stall_R], axis=1)   #zgradimo state vector X

    height = df.get("Height above ground [m]", pd.Series([0]*len(x))).to_numpy()

    observations = np.stack([x, y, z, height], axis=1)   #zgradimo observation vector Y

    #zone_feature_avgs = []
    values = []
    cols = []

    for feature in wind_features:
        for sensor_numb in range(12):
            sensor = f"W{sensor_numb + 1}"
            cols.append(f"{sensor}_{feature}")
            #print(cols)
    for col in cols:
        val = df.get(col, pd.Series([0]*len(x))).to_numpy() 
        values.append(val)
    
    controls = np.stack(values, axis=1)  # shape: (time_steps, 12)

    #for feature in wind_features:
    #    for zone, sensors in zones.items():
    #        cols = [f"{sensor}_{feature}" for sensor in sensors if f"{sensor}_{feature}" in df.columns]
    #        avg_feature = df[cols].mean(axis=1).to_numpy()
    #        zone_feature_avgs.append(avg_feature)
    
    #controls = np.stack(zone_feature_avgs, axis=1)  # shape: (time_steps, 12)
    
    return states, observations, controls


In [9]:
import os
import pandas as pd
import numpy as np

normalized_folder = r'C:\Users\vsi\Desktop\ijs\smucarski_skoki\project\simulation\2024_03_Planica_12_winds\cleaned\cleaned_quad\normalized_quad'
combined_output = os.path.join(normalized_folder, "combined_dataset.npz")
file_names = []

X_list, Y_list, U_list = [], [], []

for filename in os.listdir(normalized_folder):
    if filename.endswith('.csv'):
        file_path = os.path.join(normalized_folder, filename)
        df = pd.read_csv(file_path)

        X_state, Y_obs, U_ctrl = preprocess_flight_normalized(df)

        X_list.append(X_state)
        Y_list.append(Y_obs)
        U_list.append(U_ctrl)
        file_names.append(filename)

        #if np.any(np.isnan(X_state)):
        #    print(f"NaN values in {filename}")

        print(f"Processed {filename} -> X:{X_state.shape}, Y:{Y_obs.shape}, U:{U_ctrl.shape}")

# Save combined dataset
#np.savez(combined_output, X=np.array(X_list, dtype=object), Y=np.array(Y_list, dtype=object), U=np.array(U_list, dtype=object))
print(f"\nCombined dataset saved to {combined_output}")


AttributeError: partially initialized module 'pandas' has no attribute 'core' (most likely due to a circular import)

In [8]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from scipy.interpolate import interp1d
import os
import matplotlib.pyplot as plt


def fit_ssm_cv(X_list, U_list, Y_list, n_splits=5, alpha=1e-3):
    """
    Fit an SSM model using K-fold cross-validation across jumps.
    Uses arc-length interpolation for error computation.
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    train_errors, test_errors = [], []
    models = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_list)):
        # --- build training data ---
        X_t = np.vstack([seq[:-1] for i, seq in enumerate(X_list) if i in train_idx])
        X_next = np.vstack([seq[1:] for i, seq in enumerate(X_list) if i in train_idx])
        U_t = np.vstack([u[:-1] for i, u in enumerate(U_list) if i in train_idx])
        Y_t = np.vstack([y[:-1] for i, y in enumerate(Y_list) if i in train_idx])

        # Regression [X_next] ~ [X_t | U_t]
        Phi = np.hstack([X_t, U_t])
        ridge = Ridge(alpha=alpha, fit_intercept=False)
        ridge.fit(Phi, X_next)
        Theta = ridge.coef_.T

        state_dim = X_t.shape[1]
        A = Theta[:state_dim, :].T
        B = Theta[state_dim:, :].T

        # Regression [Y_t] ~ [X_t | U_t]
        Phi_y = np.hstack([X_t, U_t])
        ridge_y = Ridge(alpha=alpha, fit_intercept=False)
        ridge_y.fit(Phi_y, Y_t)
        Theta_y = ridge_y.coef_.T

        obs_dim = Y_t.shape[1]
        C = Theta_y[:state_dim, :].T
        D = Theta_y[state_dim:, :].T

        models.append((A, B, C, D))

        # --- training error (shape-aligned) ---
        train_sim = X_t @ A.T + U_t @ B.T
        train_err, _ = error_fun(X_next[:, :3], train_sim[:, :3])
        train_errors.append(train_err)

        # --- test error ---
        test_errs = []
        for j in test_idx:
            x0 = X_list[j][0]
            U_seq = U_list[j]
            X_seq = X_list[j]

            # simulate with learned model
            X_sim = [x0]
            for t in range(len(U_seq) - 1):
                x_next = A @ X_sim[-1] + B @ U_seq[t]
                X_sim.append(x_next)
            X_sim = np.array(X_sim)

            # compute error
            test_err, _ = error_fun(X_seq[:, :3], X_sim[:, :3])
            test_errs.append(test_err)

            # plot each trajectory
            plot(X_seq[:, :3], X_sim[:, :3], j, "2-SSM-2", test_err)

        test_errors.append(np.mean(test_errs))

        print(train_err, np.mean(test_errs))

    return np.mean(train_errors), np.mean(test_errors), models


KeyboardInterrupt: 

In [20]:
avg_train_err, avg_test_err, models = fit_ssm_cv(X_list, U_list, Y_list, n_splits=len(X_list), alpha=10)

print("average train error:", avg_train_err)
print("average test_error:", avg_test_err)

Fold 1: train_err=8.475, test_err=1.523
Fold 2: train_err=8.335, test_err=2.783
Fold 3: train_err=8.319, test_err=3.255
Fold 4: train_err=7.983, test_err=1.432
Fold 5: train_err=8.452, test_err=1.826
Fold 6: train_err=8.074, test_err=8.716
Fold 7: train_err=7.993, test_err=2.219
Fold 8: train_err=7.840, test_err=6.530
Fold 9: train_err=8.316, test_err=2.851
Fold 10: train_err=8.001, test_err=3.126
Fold 11: train_err=8.355, test_err=7.065
Fold 12: train_err=8.039, test_err=3.324
Fold 13: train_err=8.180, test_err=2.654
Fold 14: train_err=8.104, test_err=5.933
Fold 15: train_err=8.294, test_err=3.341
Fold 16: train_err=8.153, test_err=5.199
Fold 17: train_err=7.924, test_err=3.448
Fold 18: train_err=8.490, test_err=4.782
Fold 19: train_err=8.441, test_err=7.299
Fold 20: train_err=8.636, test_err=3.226
Fold 21: train_err=8.296, test_err=5.002
Fold 22: train_err=8.104, test_err=4.410
Fold 23: train_err=8.292, test_err=6.653
Fold 24: train_err=8.241, test_err=4.117
Fold 25: train_err=8.430,

c:\Users\vsi\Desktop\ijs\smucarski_skoki\project\flight_env\Lib\site-packages\scipy\interpolate\_interpolate.py:497: RuntimeWarning: invalid value encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]


Fold 134: train_err=8.091, test_err=nan
Fold 135: train_err=8.208, test_err=9.083
Fold 136: train_err=7.936, test_err=5.345
Fold 137: train_err=8.384, test_err=4.480
Fold 138: train_err=8.287, test_err=1.883
Fold 139: train_err=8.063, test_err=3.112
Fold 140: train_err=9.145, test_err=2.177
Fold 141: train_err=8.304, test_err=10.243
Fold 142: train_err=8.352, test_err=3.286
Fold 143: train_err=8.192, test_err=5.425
Fold 144: train_err=8.394, test_err=2.293
Fold 145: train_err=8.057, test_err=2.887
Fold 146: train_err=8.046, test_err=3.319
Fold 147: train_err=8.088, test_err=0.839
Fold 148: train_err=8.121, test_err=5.388
Fold 149: train_err=8.458, test_err=7.121
Fold 150: train_err=8.299, test_err=2.603
Fold 151: train_err=4.239, test_err=7.763
Fold 152: train_err=8.312, test_err=6.135
Fold 153: train_err=8.258, test_err=4.283
Fold 154: train_err=8.316, test_err=1.044
Fold 155: train_err=8.300, test_err=4.587
Fold 156: train_err=8.079, test_err=2.743
Fold 157: train_err=8.271, test_err